# C. elegans Annotation Viewer

Opens zarr image data in napari with clipping planes, StarryNite nuclei annotations,
and a lineage tree view.

Uses the `celegans_annotator` package — edit the functions there and they'll
auto-reload here thanks to `%autoreload`.

In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from pathlib import Path

import napari

from celegans_annotator import add_lineage_view, load_nuclei_from_zip
from celegans_annotator.clipping_planes import (
    add_clipping_plane_widgets,
    init_clipping_planes,
)

## Set your data paths here

Edit these paths to point to your data. Use `Path(r"...")` on Windows.

In [ ]:
# New CND-1
green_zarr_path = Path(
    r"C:\Users\yarbroughb\OneDrive - Howard Hughes Medical Institute\shroff\Lineaging Napari 3D viewer\green_channel.zarr"
)
red_zarr_path = Path(
    r"C:\Users\yarbroughb\OneDrive - Howard Hughes Medical Institute\shroff\Lineaging Napari 3D viewer\red_channel.zarr"
)
sn_zip_path = Path(
    r"Z:\shrofflab\CND-1_RedUntwisting_A\Lineaging\20260225\20260225_CND-1_lineage\Pos0\SPIMB\For_Deep_Learning\For_Lineaging\StarryNite\SN_files\Decon_emb1_edited.zip"
)

## Load StarryNite data

In [ ]:
nuclei_coords, nuclei_labels = load_nuclei_from_zip(sn_zip_path)
print(f"Loaded {len(nuclei_coords)} nuclei, {len(set(nuclei_labels))} unique cell names")

## Create napari viewer

In [ ]:
viewer = napari.Viewer()

green = viewer.open(
    path=str(green_zarr_path),
    blending="additive",
    contrast_limits=(0, 300),
    colormap="green",
    rendering="attenuated_mip",
    attenuation=0.75,
)
red = viewer.open(
    path=str(red_zarr_path),
    blending="additive",
    contrast_limits=(0, 300),
    colormap="red",
    rendering="attenuated_mip",
    attenuation=0.75,
)

viewer.add_points(
    nuclei_coords,
    ndim=4,
    opacity=0.7,
    size=6,
    face_color="transparent",
    border_color="cyan",
    properties={"name": nuclei_labels},
    text="name",
    blending="additive",
)

green_layer = viewer.layers[0]
red_layer = viewer.layers[1]
point_layer = viewer.layers[2]

## Add clipping planes

In [ ]:
# Get spatial dimensions from the image data (t, x, y, z)
t, x, y, z = green_layer.data.shape
layers = [green_layer, red_layer, point_layer]

init_clipping_planes(layers, shape=(x, y, z))
add_clipping_plane_widgets(viewer, layers, shape=(x, y, z))

viewer.dims.ndisplay = 3

## Add lineage tree view

In [ ]:
add_lineage_view(viewer, nuclei_coords, nuclei_labels)

In [ ]:
napari.run()